# 04 · Generalized additive model (GAM)

**Question:** can a GAM capture non-linear relationships between state-level factors and early-onset (ages 20–49) CRC incidence, and how does it compare with the original three models?

**Outcome:** state-year crude CRC rate among adults aged 20–49, per 100,000.

**Features:** year, state baseline early-onset rate, red meat and poultry availability, adult smoking, adult obesity, uninsured rate, log population, Census region.

**Validation:**
- 5-fold **GroupKFold by state**, matching the original modeling notebook.
- **Temporal holdouts:** train through 2016 and test on 2017–2019; separately, train through 2019 and test on 2020.

**Model:** additive cubic splines with Ridge regularization (6 knots, alpha = 10). This adds a non-linear model to the Ridge, random forest, and histogram gradient boosting (HGB) comparison without changing the original notebooks.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

def _find_project_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "data_prep.py").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Open this notebook inside the full GitHub repository."
    )

ROOT = _find_project_root()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mpl"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer

from src.model import load_data

CLEAN = ROOT / "outputs" / "cleaned"
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# Match the feature list in the ORIGINAL src/model.py.
NUMERIC = [
    "year",
    "crc_early_rate_baseline",  # kept only for a like-for-like original comparison
    "red_meat_lb_pc",
    "poultry_lb_pc",
    "smoking_pct",
    "obesity_pct",
    "uninsured_pct",
    "log_population",
]
CATEGORICAL = ["region"]
TARGET = "crc_early_rate"

source_path = CLEAN / "analysis_panel_1999_2020.csv"
if not source_path.exists():
    raise FileNotFoundError(
        f"Missing {source_path}. Run 01_data_cleaning.ipynb first."
    )

df = load_data()
X = df[NUMERIC + CATEGORICAL].copy()
y = df[TARGET].copy()
groups = df["state_abbr"]

print(f"Project root: {ROOT}")
print(f"Outcome rows: {len(df):,}; states: {groups.nunique()}; years: {df.year.min()}–{df.year.max()}")
print("Predictors:", NUMERIC + CATEGORICAL)


## 1. GAM performance

The GAM fits a separate smooth curve for each numeric predictor and treats Census region as categorical. Preprocessing is fitted within each training fold. The spline settings are fixed for this initial comparison rather than tuned for optimal performance.

In [ ]:
def make_gam():
    spline_blocks = [
        (
            f"smooth_{feature}",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("spline", SplineTransformer(
                    n_knots=6,
                    degree=3,
                    include_bias=False,
                    extrapolation="constant",
                )),
            ]),
            [feature],
        )
        for feature in NUMERIC
    ]
    prep = ColumnTransformer(
        spline_blocks + [
            ("region", OneHotEncoder(
                drop="first", handle_unknown="ignore", sparse_output=False
            ), CATEGORICAL),
        ],
        remainder="drop",
        sparse_threshold=0,
    )
    return Pipeline([
        ("prep", prep),
        ("gam", Ridge(alpha=10.0)),
    ])

print(make_gam())


In [ ]:
gam_name = "GAM (penalized splines)"

# GroupKFold: same grouping and number of folds as the original notebook.
gam_oof_pred = cross_val_predict(
    make_gam(), X, y, groups=groups,
    cv=GroupKFold(n_splits=5), n_jobs=1,
)

results = [{
    "model": gam_name,
    "validation": "5-fold GroupKFold (by state)",
    "r2": r2_score(y, gam_oof_pred),
    "mae": mean_absolute_error(y, gam_oof_pred),
    "n": len(y),
}]

for label, train_mask, test_mask in [
    ("Temporal holdout 2017-2019", df.year <= 2016, df.year.between(2017, 2019)),
    ("Temporal holdout 2020", df.year <= 2019, df.year == 2020),
]:
    model = make_gam()
    model.fit(X.loc[train_mask], y.loc[train_mask])
    predictions = model.predict(X.loc[test_mask])
    results.append({
        "model": gam_name,
        "validation": label,
        "r2": r2_score(y.loc[test_mask], predictions),
        "mae": mean_absolute_error(y.loc[test_mask], predictions),
        "n": int(test_mask.sum()),
    })

gam_metrics = pd.DataFrame(results).round(3)
display(gam_metrics)

# New filenames: no original output files are overwritten.
gam_metrics.to_csv(CLEAN / "gam_model_metrics.csv", index=False)
gam_group_predictions = df[["state_abbr", "year", TARGET]].copy()
gam_group_predictions["gam_prediction"] = gam_oof_pred
gam_group_predictions.to_csv(CLEAN / "gam_groupcv_predictions.csv", index=False)
print("Saved: gam_model_metrics.csv and gam_groupcv_predictions.csv")


**Reading the metrics.** The GAM reaches **R² = 0.496** (MAE **1.071**) in state-grouped cross-validation and **R² = 0.411** (MAE **1.049**) for 2017–2019. Performance is lower for 2020 (**R² = 0.092**, MAE **1.177**). MAE is measured in cases per 100,000; lower is better. The weaker 2020 result does not, by itself, identify the reason for the decline.

## 2. Comparison with the original models

HGB has the highest temporal R² in both 2017–2019 (**0.448**) and 2020 (**0.245**). GAM is competitive in 2017–2019, with the lowest MAE (**1.049**), although the difference from random forest (**1.052**) and HGB (**1.053**) is very small. Ridge has the highest GroupKFold R² (**0.537**). Overall, HGB remains the strongest choice on future-year R², while GAM offers an interpretable non-linear alternative.

The code below combines GAM results with `model_metrics.csv` from the original modeling notebook. It does not overwrite those results.

In [ ]:
original_metrics_path = CLEAN / "model_metrics.csv"
if original_metrics_path.exists():
    original_metrics = pd.read_csv(original_metrics_path)
    combined = pd.concat([original_metrics, gam_metrics], ignore_index=True)
    print("R² comparison (higher is better):")
    display(combined.pivot(index="model", columns="validation", values="r2").round(3))
    print("MAE comparison (lower is better):")
    display(combined.pivot(index="model", columns="validation", values="mae").round(3))
else:
    print("Original model_metrics.csv not found. Run the original 03_modeling.ipynb to compare all four models.")


## 3. Combined 2017–2020 holdout

For a direct comparison with the original report figure, the GAM is trained through 2016 and then predicts all four years, **2017–2020**, without refitting. This is different from the separate 2020 test above, which trains through 2019.

In [ ]:
train_mask = df.year <= 2016
test_mask = df.year >= 2017

gam_2016 = make_gam()
gam_2016.fit(X.loc[train_mask], y.loc[train_mask])

holdout_pred = gam_2016.predict(X.loc[test_mask])
gam_holdout = df.loc[test_mask, ["state_abbr", "year", TARGET]].copy()
gam_holdout["prediction"] = holdout_pred
gam_holdout["residual"] = gam_holdout[TARGET] - gam_holdout["prediction"]
gam_holdout.to_csv(CLEAN / "gam_test_predictions_2017_2020.csv", index=False)

print(f"GAM 2017–2020 holdout R²: {r2_score(y.loc[test_mask], holdout_pred):.3f}")
print(f"GAM 2017–2020 holdout MAE: {mean_absolute_error(y.loc[test_mask], holdout_pred):.3f}")


**Reading the holdout.** The combined 2017–2020 evaluation gives **R² = 0.331** and **MAE = 1.086**. The GAM captures part of the variation in early-onset CRC incidence, but predicted rates still differ from observed rates by about **1.09 cases per 100,000** on average.

## 4. What do the fitted responses show?

The first plot compares observed and predicted incidence in the 2017–2020 holdout. The other panels vary obesity prevalence or red meat availability while holding the remaining predictors at representative values. These are **fitted model responses**, not estimated causal effects.

In [ ]:
# Create a representative observation using only training-period predictor values.
reference = X.loc[train_mask].copy()
base = {feature: float(reference[feature].median()) for feature in NUMERIC}
base["region"] = str(reference["region"].mode().iloc[0])

def response_curve(feature, points=100):
    nonmissing = reference[feature].dropna()
    grid = np.linspace(nonmissing.quantile(0.05), nonmissing.quantile(0.95), points)
    sample = pd.DataFrame([base] * points, columns=NUMERIC + CATEGORICAL)
    sample[feature] = grid
    return grid, gam_2016.predict(sample)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

axes[0].scatter(gam_holdout[TARGET], gam_holdout["prediction"], s=17, alpha=0.7)
low = min(gam_holdout[TARGET].min(), gam_holdout["prediction"].min())
high = max(gam_holdout[TARGET].max(), gam_holdout["prediction"].max())
axes[0].plot([low, high], [low, high], "--", color="gray", lw=1)
axes[0].set_xlabel("Observed CRC rate / 100,000")
axes[0].set_ylabel("Predicted CRC rate / 100,000")
axes[0].set_title("GAM: 2017–2020 predictions")

for ax, col, label in [
    (axes[1], "obesity_pct", "Obesity prevalence (%)"),
    (axes[2], "red_meat_lb_pc", "Red meat availability (lb/person)"),
]:
    xx, yy = response_curve(col)
    ax.plot(xx, yy, lw=2)
    ax.set_xlabel(label)
    ax.set_ylabel("Predicted CRC rate / 100,000")
    ax.set_title(f"Fitted GAM response: {col}")
    ax.grid(alpha=0.2)

fig.suptitle("GAM results (associations only; other variables held fixed)", y=1.03)
fig.tight_layout()
figure_path = FIGURES / "fig5_gam_results.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", figure_path)


**Figure 5.** (a) Observed versus predicted early-onset CRC rates; the dashed line is perfect prediction. Predictions concentrate near the middle of the observed range, so the model tends to overpredict some low rates and underpredict some high rates. (b) The fitted response increases overall as obesity prevalence rises, from roughly **11.5 to 12.7** cases per 100,000 across the displayed range. (c) The fitted response for red meat availability is non-linear, rising near **149–150 lb/person** and falling at higher values.

The red meat curve should not be interpreted as evidence of a protective effect at higher availability levels: meat availability is measured nationally and varies with year, rather than between states within the same year.

## 5. Interpretation and limitations

- **Model comparison.** GAM performs below HGB on future-year R² but gives a similar 2017–2019 MAE. Its main added value is showing smooth, non-linear model responses.
- **Prediction, not causation.** Obesity and red meat curves describe associations in the fitted model; they do not establish changes in individual cancer risk.
- **National exposure measure.** USDA meat availability is not state-specific consumption or processed-meat intake, so it is difficult to distinguish its signal from other national time trends.
- **Original baseline retained.** For consistency with `03_modeling.ipynb`, the GAM uses `crc_early_rate_baseline`. Because it comes from observed state outcomes, GroupKFold is **not** a clean test for states with no known historical rates. This can be revisited with the group.
- **Fixed model settings.** The spline and regularization parameters were not tuned, so the results are an initial comparison rather than an optimized GAM.

**Bottom line:** HGB remains the strongest model on the temporal R² comparisons. GAM is worth retaining as a fourth model because its non-linear response curves provide a different view of the predictors, with the same limitations as the original data and validation design.

**Outputs:** `gam_model_metrics.csv`, `gam_groupcv_predictions.csv`, and `gam_test_predictions_2017_2020.csv` in `outputs/cleaned/`; `fig5_gam_results.png` in `figures/`. The original notebooks and their output files are not modified.